# Truco SFT Agent GRPO Training on Google Colab

This notebook trains a Qwen-1.5B model to play **Truco Paulista** using Group Relative Policy Optimization (GRPO) on our public dataset `manzoliw/trucobench-sft`.

It integrates **Weights & Biases (W&B)** to plot live training graphs (rewards, KL divergence, loss) in your browser.

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone repo to absolute path /content/trucobench if not already cloned
import os
if not os.path.exists("/content/trucobench"):
    !git clone https://github.com/ManzoliW/trucobench.git /content/trucobench

# Change directory to the absolute path to prevent duplication errors on re-run
%cd /content/trucobench

# Force upgrade all dependencies to the latest versions (essential for GRPOTrainer support)
!pip install -U -q trl peft bitsandbytes accelerate datasets transformers wandb

### Step 2: Configure Secrets and Run GRPO Training

To enable model downloading and live training charts:
1. Go to your **Hugging Face tokens** page at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and copy your Write Token.
2. Go to your **W&B authorize** page at [wandb.ai/authorize](https://wandb.ai/authorize) and copy your API Key (create a free account if you don't have one).
3. Open Colab's **Secrets (key icon)** in the left sidebar.
4. Add two secrets and toggle **Notebook access** to ON for both:
   - **`HF_TOKEN`**: *Your Hugging Face write token*
   - **`WANDB_API_KEY`**: *Your Weights & Biases API Key*

In [ ]:
import os
from google.colab import userdata

# Inject Hugging Face token
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ Hugging Face Token loaded successfully!")
except Exception as e:
    print("❌ Error loading Hugging Face token. Ensure HF_TOKEN secret is configured and access is enabled in the Secrets tab.")

# Inject W&B API Key for live charts
try:
    os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
    os.environ["WANDB_PROJECT"] = "truco-grpo"
    print("✅ Weights & Biases API Key loaded successfully!")
except Exception as e:
    print("⚠️ Warning: WANDB_API_KEY secret not found. Training will log to local TensorBoard instead of W&B.")

# Determine reporting framework
report_framework = "wandb" if os.environ.get("WANDB_API_KEY") else "tensorboard"

# Execute GRPO training
!python scripts/train-grpo.py \
    --model_id "Qwen/Qwen2.5-1.5B-Instruct" \
    --load_in_4bit \
    --batch_size 1 \
    --gradient_accumulation_steps 8 \
    --group_size 4 \
    --dataset_size 10000 \
    --report_to {report_framework} \
    --output_dir "./output/truco-grpo"

### Step 3: Compress and Download Trained Weights

Run the cell below to compress your LoRA adapters and download them to your local computer.

In [ ]:
from google.colab import files
import shutil

# Zip adapter output directory
shutil.make_archive("truco_grpo_adapter", "zip", "./output/truco-grpo")

# Download the zip file
files.download("truco_grpo_adapter.zip")